In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

eurocontrol_raw = pd.read_csv("../data/raw/eurocontrol_punctuality.csv")
ourairports_raw = pd.read_csv("../data/raw/ourairports_airports.csv")

print("=" * 60)
print("DATASETS LOADED SUCCESSFULLY")
print("=" * 60)

print(f"\nEUROCONTROL shape : {eurocontrol_raw.shape}")
print(f"OurAirports shape : {ourairports_raw.shape}")

print("\nEUROCONTROL columns:")
print(eurocontrol_raw.columns.tolist())

print("\nOurAirports columns:")
print(ourairports_raw.columns.tolist())

display(eurocontrol_raw.head(3))


In [ ]:

TARGET_ISO = ["BE", "DE", "ES", "FR", "IT"]

def norm(s):
    if pd.isna(s):
        return None
    s = str(s).upper().strip()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

euro = eurocontrol_raw.copy()
air = ourairports_raw.copy()

euro["airport_norm"] = euro["Airport"].apply(norm)

air["name_norm"] = air["name"].apply(norm)
air["municipality_norm"] = air["municipality"].apply(norm)

air_target = air[air["iso_country"].isin(TARGET_ISO)].copy()

KEEP_COLS = [
    "id","name","municipality","iso_country","iata_code","ident",
    "latitude_deg","longitude_deg","elevation_ft","type","scheduled_service"
]

unique_euro = pd.DataFrame({
    "airport_norm": euro["airport_norm"].dropna().unique()
})

m1 = unique_euro.merge(
    air_target[KEEP_COLS + ["municipality_norm"]],
    left_on="airport_norm",
    right_on="municipality_norm",
    how="left"
)

unmatched = (
    m1[m1["id"].isna()][["airport_norm"]]
    .drop_duplicates()
)

air_small = (
    air_target[KEEP_COLS + ["name_norm"]]
    .dropna(subset=["name_norm"])
    .copy()
)

matches = []

for airport_name in unmatched["airport_norm"].dropna().unique():
    tmp = air_small[
        air_small["name_norm"].str.contains(
            rf"\b{re.escape(airport_name)}\b",
            regex=True,
            na=False
        )
    ].copy()

    tmp["airport_norm"] = airport_name
    matches.append(tmp)

if matches:
    m2 = pd.concat(matches, ignore_index=True)
else:
    m2 = pd.DataFrame()

cand = pd.concat(
    [m1[m1["id"].notna()], m2],
    ignore_index=True
)

type_rank = {
    "large_airport": 3,
    "medium_airport": 2,
    "small_airport": 1
}

cand["type_score"] = cand["type"].map(type_rank).fillna(0)

cand["sched_score"] = (
    cand["scheduled_service"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

cand["iata_score"] = cand["iata_code"].notna().astype(int)

cand["score"] = (
    cand["type_score"] * 10 +
    cand["sched_score"] * 3 +
    cand["iata_score"] * 2
)

best = (
    cand
    .sort_values(["airport_norm", "score"], ascending=[True, False])
    .drop_duplicates("airport_norm")
    [["airport_norm"] + KEEP_COLS]
)

df = euro.merge(best, on="airport_norm", how="left")

print(f"Match rate : {df['id'].notna().mean():.1%}")
print(f"df shape   : {df.shape}")

display(df.head(5))


In [ ]:

df5 = (
    df[df["iso_country"].isin(TARGET_ISO)]
    .copy()
    .reset_index(drop=True)
)

date_col = "Date"

df5[date_col] = pd.to_datetime(df5[date_col], errors="coerce")

def pct_to_float(series):

    def _convert(v):

        if pd.isna(v):
            return np.nan

        s = str(v).strip()

        if s.endswith("%"):
            try:
                return float(s[:-1])
            except:
                return np.nan

        try:
            f = float(s)

            if 0 < f <= 1:
                return f * 100

            return f

        except:
            return np.nan

    return series.apply(_convert)

PCT_COLS = [
    "Departure Punctuality %",
    "Arrival Punctuality %",
    "Operated Schedules %"
]

for col in PCT_COLS:
    if col in df5.columns:
        df5[col] = pct_to_float(df5[col])

DELAY_COLS = [
    "Avg Departure Schedule Delay",
    "Avg Arrival Schedule Delay",
    "Avg Departure - Arrival Schedule Delay"
]

for col in DELAY_COLS:
    if col in df5.columns:
        df5[col] = pd.to_numeric(df5[col], errors="coerce")

main_metric = "Departure Punctuality %"

rows_before = len(df5)

df5 = df5.dropna(subset=[main_metric])

df5 = df5.drop_duplicates(subset=["airport_norm", date_col])

if "elevation_ft" in df5.columns:
    df5["elevation_ft"] = (
        df5["elevation_ft"]
        .fillna(df5["elevation_ft"].median())
    )

if "scheduled_service" in df5.columns:
    df5["scheduled_service"] = (
        df5["scheduled_service"]
        .fillna("unknown")
    )

COUNTRY_NAMES = {
    "BE": "Belgium",
    "DE": "Germany",
    "ES": "Spain",
    "FR": "France",
    "IT": "Italy"
}

df5["year"] = df5[date_col].dt.year
df5["month"] = df5[date_col].dt.month
df5["month_name"] = df5[date_col].dt.strftime("%b")
df5["weekday"] = df5[date_col].dt.dayofweek
df5["weekday_name"] = df5[date_col].dt.strftime("%a")
df5["is_weekend"] = (df5["weekday"] >= 5).astype(int)
df5["quarter"] = df5[date_col].dt.quarter

season_map = {
    12: "Winter",
     1: "Winter",
     2: "Winter",
     3: "Spring",
     4: "Spring",
     5: "Spring",
     6: "Summer",
     7: "Summer",
     8: "Summer",
     9: "Autumn",
    10: "Autumn",
    11: "Autumn"
}

df5["season"] = df5["month"].map(season_map)

df5["country_name"] = df5["iso_country"].map(COUNTRY_NAMES)

def delay_cat(v):
    if pd.isna(v):
        return "Unknown"
    if v >= 80:
        return "Good"
    elif v >= 65:
        return "Moderate"
    elif v >= 50:
        return "Poor"
    else:
        return "Critical"

df5["delay_category"] = df5[main_metric].apply(delay_cat)

def classify_airport(t):
    t = str(t).lower()
    if "large" in t:
        return "Hub"
    elif "medium" in t:
        return "Regional"
    else:
        return "Small"

df5["traffic_class"] = df5["type"].apply(classify_airport)

df5["arrival_vs_departure"] = (
    df5["Arrival Punctuality %"] -
    df5["Departure Punctuality %"]
)

df5["metric_vs_mean"] = (
    df5[main_metric] -
    df5[main_metric].mean()
)

df5["above_avg"] = (
    df5["metric_vs_mean"] > 0
).astype(int)

df5["year_quarter"] = (
    df5["year"].astype(str) +
    "-Q" +
    df5["quarter"].astype(str)
)

print(f"Total columns : {df5.shape[1]}")
print(f"Total rows    : {len(df5):,}")

display(df5.head())
